# Databricks/PySpark dla Data Analyst/Engineer — Moduł 6: Wczytywanie i zapisywanie danych

Do tej pory nasz zestaw danych powstawał w pamięci (z `pandas` przez `createDataFrame`).
W praktyce dane niemal zawsze zaczynają i kończą swoją drogę w PLIKACH -- ten moduł
pokazuje, jak Spark je zapisuje i wczytuje: CSV, JSON i Parquet, tryby zapisu oraz
partycjonowanie.

## Spis treści
1. [Zapis do plików: CSV, JSON, Parquet](#sec1)
2. [Odczyt: schemat jawny kontra wnioskowany](#sec2)
3. [Tryby zapisu (write modes)](#sec3)
4. [Partycjonowanie przy zapisie](#sec4)
5. [Podsumowanie i ćwiczenia](#sec5)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("kurs_spark").getOrCreate()
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


<a id="sec1"></a>
## 1. Zapis do plików: CSV, JSON, Parquet

`df.write.csv(...)`, `df.write.json(...)`, `df.write.parquet(...)` -- zapisują DataFrame
do plików. WAŻNE: Spark zapisuje nie JEDEN plik, tylko CAŁY FOLDER, wewnątrz którego
znajduje się po jednym pliku na każdą partycję (plus plik `_SUCCESS`) -- to naturalna
konsekwencja rozproszonego przetwarzania: każdy executor zapisuje swój kawałek osobno.

In [ ]:
import os

sciezka_bazowa = "dane_wyjsciowe"
os.makedirs(sciezka_bazowa, exist_ok=True)

produkty.write.mode("overwrite").csv(f"{sciezka_bazowa}/produkty_csv", header=True)
print(os.listdir(f"{sciezka_bazowa}/produkty_csv"))


In [ ]:
produkty.write.mode("overwrite").json(f"{sciezka_bazowa}/produkty_json")
produkty.write.mode("overwrite").parquet(f"{sciezka_bazowa}/produkty_parquet")
print("Zapisano JSON i Parquet.")


> ⚡ **Dlaczego Parquet jest domyślnym wyborem w Sparku**
>
> **Parquet** to format kolumnowy (nie wierszowy jak CSV) z wbudowanym schematem i kompresją -- Spark musi wczytać TYLKO te kolumny, które faktycznie są w zapytaniu (tzw. *predicate/column pruning*), i nie musi zgadywać typów, bo są zapisane razem z danymi. W praktyce zawodowej Parquet (a w Databricks -- Delta Lake, nadbudowany właśnie nad Parquetem -- zobacz Moduł 10) to domyślny wybór dla większości pipeline'ów, CSV/JSON głównie jako format wymiany z systemami zewnętrznymi.

<a id="sec2"></a>
## 2. Odczyt: schemat jawny kontra wnioskowany

Przy odczycie CSV/JSON Spark domyślnie NIE zna typów kolumn -- trzeba mu jawnie kazać je
odgadnąć (`inferSchema=True`, kosztowne -- podwójny odczyt danych, jak wspomnieliśmy w
Module 2) albo podać JAWNY schemat z góry (szybsze, przewidywalne).

In [ ]:
# Wnioskowanie schematu -- wygodne, ale kosztowne na duzych danych
produkty_wywnioskowane = spark.read.csv(
    f"{sciezka_bazowa}/produkty_csv", header=True, inferSchema=True
)
produkty_wywnioskowane.printSchema()


In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schemat_produktow = StructType([
    StructField("produkt_id", IntegerType(), False),
    StructField("kategoria", StringType(), True),
    StructField("cena", DoubleType(), True),
])

produkty_jawny_schemat = spark.read.csv(
    f"{sciezka_bazowa}/produkty_csv", header=True, schema=schemat_produktow
)
produkty_jawny_schemat.printSchema()


> ⚠️ **Parquet NIE ma tego problemu**
>
> Parquet zapisuje schemat RAZEM z danymi -- przy odczycie `spark.read.parquet(...)` Spark zawsze zna dokładne typy kolumn bez żadnego zgadywania czy jawnego podawania schematu. To kolejny (obok wydajności) powód, dla którego Parquet jest wygodniejszy niż CSV w większości potoków danych.

In [ ]:
produkty_z_parquet = spark.read.parquet(f"{sciezka_bazowa}/produkty_parquet")
produkty_z_parquet.printSchema()


<a id="sec3"></a>
## 3. Tryby zapisu (write modes)

Parametr `.mode(...)` mówi Sparkowi, co zrobić, jeśli docelowy folder JUŻ ISTNIEJE:

- `"overwrite"` -- usuwa istniejące dane i zapisuje od nowa (używaliśmy tego wyżej),
- `"append"` -- dopisuje nowe pliki obok istniejących,
- `"error"` (domyślny, gdy nic nie podamy) -- rzuca wyjątek, jeśli folder już istnieje,
- `"ignore"` -- jeśli folder istnieje, po prostu nic nie robi (cicho pomija zapis).

In [ ]:
# append -- dopisujemy kolejna porcje danych do tego samego folderu
pierwsza_polowa = klienci.filter(klienci.klient_id <= 150)
druga_polowa = klienci.filter(klienci.klient_id > 150)

pierwsza_polowa.write.mode("overwrite").parquet(f"{sciezka_bazowa}/klienci_parquet")
druga_polowa.write.mode("append").parquet(f"{sciezka_bazowa}/klienci_parquet")

wczytani = spark.read.parquet(f"{sciezka_bazowa}/klienci_parquet")
print(f"Liczba wierszy po append: {wczytani.count()}")


> ⚠️ **error i ignore łatwo pomylić z brakiem błędu**
>
> Jeśli zapomnisz o `.mode(...)` i folder już istnieje, dostaniesz wyjątek (`AnalysisException: path already exists`) -- to bezpieczne domyślne zachowanie, chroniące przed przypadkowym nadpisaniem danych. `"ignore"` jest rzadziej używany i łatwo o nim zapomnieć -- upewnij się, że naprawdę chcesz CICHO pominąć zapis, zanim go użyjesz.

<a id="sec4"></a>
## 4. Partycjonowanie przy zapisie

`partitionBy("kolumna")` dzieli zapisywane dane na PODFOLDERY według wartości kolumny
(np. `kategoria=Elektronika/`, `kategoria=Odziez/`, ...). Dzięki temu późniejszy odczyt z
FILTREM na tej kolumnie może pominąć całe podfoldery, zamiast czytać wszystko
(*partition pruning*) -- bardzo ważne dla wydajności na dużych zbiorach danych.

In [ ]:
produkty.write.mode("overwrite").partitionBy("kategoria").parquet(
    f"{sciezka_bazowa}/produkty_partycjonowane"
)
for folder in sorted(os.listdir(f"{sciezka_bazowa}/produkty_partycjonowane")):
    print(folder)


In [ ]:
# Odczyt z filtrem na skolumnie partycjonujacej -- Spark pominie niepotrzebne podfoldery
produkty_elektronika = spark.read.parquet(
    f"{sciezka_bazowa}/produkty_partycjonowane"
).filter("kategoria = 'Elektronika'")
produkty_elektronika.explain()


> 🧱 **Wybór kolumny partycjonującej -- z rozwagą**
>
> Dobra kolumna partycjonująca ma NIEWIELE unikalnych wartości używanych często w filtrach WHERE (np. `kategoria`, `region`, albo data obcięta do dnia/miesiąca). Partycjonowanie po kolumnie z tysiącami unikalnych wartości (np. `klient_id`) tworzy tysiące malutkich podfolderów -- to spowalnia, a nie przyspiesza. Wracamy do tego szerzej w Module 9 (optymalizacja).

<a id="sec5"></a>
## 5. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- zapis do CSV/JSON/Parquet (`df.write.csv/json/parquet(...)`) -- zawsze jako folder, nie pojedynczy plik,
- odczyt z jawnym schematem kontra wnioskowanym (`inferSchema`) -- i dlaczego Parquet omija ten problem,
- tryby zapisu: `overwrite`, `append`, `error`, `ignore`,
- partycjonowanie przy zapisie (`partitionBy`) i *partition pruning* przy odczycie.

### 📝 Ćwiczenie 1: Zapis i odczyt zamówień

Zapisz `zamowienia` do Parquet (`overwrite`), a następnie odczytaj je z powrotem i wypisz `printSchema()` oraz `count()`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
zamowienia.write.mode("overwrite").parquet("dane_wyjsciowe/zamowienia_parquet")
odczytane = spark.read.parquet("dane_wyjsciowe/zamowienia_parquet")
odczytane.printSchema()
print(f"Liczba wierszy: {odczytane.count()}")
```

</details>

### 📝 Ćwiczenie 2: Jawny schemat dla pracowników

Zapisz `pracownicy` do CSV z nagłówkiem, a potem odczytaj plik z JAWNYM schematem (`StructType`) zamiast `inferSchema`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

pracownicy.write.mode("overwrite").csv("dane_wyjsciowe/pracownicy_csv", header=True)

schemat = StructType([
    StructField("pracownik_id", IntegerType(), False),
    StructField("region", StringType(), True),
    StructField("dzial", StringType(), True),
])
spark.read.csv("dane_wyjsciowe/pracownicy_csv", header=True, schema=schemat).printSchema()
```

</details>

### 📝 Ćwiczenie 3: Append dwóch porcji

Podziel `produkty` na dwie części (`produkt_id <= 20` i `produkt_id > 20`), zapisz pierwszą przez `overwrite`, drugą przez `append` do tego samego folderu, i sprawdź końcową liczbę wierszy.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
czesc1 = produkty.filter(produkty.produkt_id <= 20)
czesc2 = produkty.filter(produkty.produkt_id > 20)

czesc1.write.mode("overwrite").parquet("dane_wyjsciowe/produkty_append_test")
czesc2.write.mode("append").parquet("dane_wyjsciowe/produkty_append_test")

wynik = spark.read.parquet("dane_wyjsciowe/produkty_append_test")
print(f"Liczba wierszy: {wynik.count()}")
```

</details>

### 📝 Ćwiczenie 4: Partycjonowanie zamówień po statusie

Zapisz `zamowienia` z `partitionBy("status")` i wypisz nazwy powstałych podfolderów.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
import os

zamowienia.write.mode("overwrite").partitionBy("status").parquet(
    "dane_wyjsciowe/zamowienia_partycjonowane"
)
for folder in sorted(os.listdir("dane_wyjsciowe/zamowienia_partycjonowane")):
    print(folder)
```

</details>

> 🔥 **Wyzwanie: pełny potok plik -> transformacja -> plik partycjonowany**
>
> Wczytaj `produkty` z Parquet zapisanego w sekcji 1, dodaj kolumnę `przedzial_cenowy` (jak w Module 3, `when/otherwise`: `'Tani'`/`'Sredni'`/`'Drogi'`), zapisz wynik partycjonowany jednocześnie po `kategoria` I `przedzial_cenowy` (`partitionBy("kategoria", "przedzial_cenowy")`), a na końcu odczytaj z powrotem i policz liczbę produktów w każdej kombinacji.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.sql import functions as F

produkty_wczytane = spark.read.parquet("dane_wyjsciowe/produkty_parquet")

produkty_z_przedzialem = produkty_wczytane.withColumn(
    "przedzial_cenowy",
    F.when(F.col("cena") < 100, "Tani")
    .when(F.col("cena") <= 500, "Sredni")
    .otherwise("Drogi"),
)

produkty_z_przedzialem.write.mode("overwrite").partitionBy(
    "kategoria", "przedzial_cenowy"
).parquet("dane_wyjsciowe/produkty_pelny_potok")

wynik = spark.read.parquet("dane_wyjsciowe/produkty_pelny_potok")
wynik.groupBy("kategoria", "przedzial_cenowy").count().orderBy(
    "kategoria", "przedzial_cenowy"
).show(20)
```

Partycjonowanie po WIELU kolumnach tworzy zagniezdzone podfoldery: kategoria=X/przedzial_cenowy=Y/ -- kazda unikalna kombinacja to osobna sciezka.

</details>

## Co dalej?

W **Module 7** przejdziemy do funkcji okienkowych (`Window`): `row_number`, `rank`,
sumy narastające i `lag`/`lead` -- odpowiednik funkcji okienkowych z Modułu 11 kursu SQL,
teraz w PySpark.